## Verify GPU

In [ ]:
import gc
gc.collect()

287

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Go to Runtime → Change runtime type → Hardware accelerator → T4 GPU, "
        "then reconnect."
    )

print(f"✅ GPU          : {torch.cuda.get_device_name(0)}")
print(f"   VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"   CUDA        : {torch.version.cuda}")
print(f"   PyTorch     : {torch.__version__}")


✅ GPU          : Tesla T4
   VRAM        : 15.6 GB
   CUDA        : 12.8
   PyTorch     : 2.10.0+cu128


## Install libraries


In [ ]:
%pip install rouge-score -q
%pip install bert-score -q
%pip install radgraph -q
%pip install nltk -q

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("✅ All packages installed.")


✅ All packages installed.


## Upload `processed_outputs.zip`

In [ ]:
import zipfile
from pathlib import Path

PROCESSED_DIR = Path("/content/iu_xray/dataset/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Update this path to wherever your zip was uploaded
zip_path = Path("/content/processed_outputs.zip")

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(PROCESSED_DIR)
    print("Extracted:", zf.namelist())

# Verify
for fname in ["train.csv", "val.csv", "test.csv", "problem_label_map.json"]:
    exists = (PROCESSED_DIR / fname).exists()
    print(f"  {'✅' if exists else '❌ MISSING'}  {fname}")

Extracted: ['train.csv', 'val.csv', 'test.csv', 'master_dataset.csv', 'problem_label_map.json']
  ✅  train.csv
  ✅  val.csv
  ✅  test.csv
  ✅  problem_label_map.json


In [ ]:
from pathlib import Path

PROJECT_DIR   = Path("/content/iu_xray")
RAW_DIR       = PROJECT_DIR / "dataset" / "raw"
PROCESSED_DIR = PROJECT_DIR / "dataset" / "processed"
IMAGES_DIR    = RAW_DIR / "images" / "images_normalized"

RAW_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from google.colab import userdata
import os, json as _json, subprocess, shutil

# ── Load Kaggle token from Colab Secrets ──────────────────────────────────────

KAGGLE_API_TOKEN = userdata.get("KAGGLE_API_TOKEN").strip()

if not KAGGLE_API_TOKEN:
    raise ValueError(
        "KAGGLE_API_TOKEN not found in Colab Secrets.\n"
        "Click the 🔑 key icon (left sidebar) → Add new secret\n"
        "  Name : KAGGLE_API_TOKEN\n"
        "  Value: your token\n"
        "  Toggle 'Notebook access' ON, then re-run this cell."
    )

os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN
os.makedirs("/root/.config/kaggle", exist_ok=True)
with open("/root/.config/kaggle/kaggle.json", "w") as f:
    _json.dump({"username": "user", "key": KAGGLE_API_TOKEN}, f)
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)
print("✅ Kaggle token loaded from Colab Secrets.")

# ── Download IU X-Ray images from Kaggle ─────────────────────────────────────
print("\nDownloading IU X-Ray images from Kaggle …")
result = subprocess.run(
    ["kaggle", "datasets", "download",
     "-d", "raddar/chest-xrays-indiana-university",
     "-p", str(RAW_DIR),
     "--unzip"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Kaggle download failed.")

# ── Locate and move image directory if needed ─────────────────────────────────
def find_dir(root, name):
    matches = [p for p in root.rglob(name) if p.is_dir()]
    if not matches:
        raise FileNotFoundError(f"Directory '{name}' not found under {root}")
    return matches[0]

images_src = find_dir(RAW_DIR, "images_normalized")
if images_src != IMAGES_DIR:
    if IMAGES_DIR.exists():
        shutil.rmtree(IMAGES_DIR)
    shutil.copytree(images_src, IMAGES_DIR)

img_count = len(list(IMAGES_DIR.glob("*.png")))
print(f"\n✅ images_normalized : {img_count} PNG files at {IMAGES_DIR}")


✅ Kaggle token loaded from Colab Secrets.

Dataset URL: https://www.kaggle.com/datasets/raddar/chest-xrays-indiana-university
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)



✅ images_normalized : 7470 PNG files at /content/iu_xray/dataset/raw/images/images_normalized


## Reconstruct full image paths in CSVs

In [ ]:
import pandas as pd

TRAIN_CSV = PROCESSED_DIR / "train.csv"
VAL_CSV   = PROCESSED_DIR / "val.csv"
TEST_CSV  = PROCESSED_DIR / "test.csv"
LABEL_MAP = PROCESSED_DIR / "problem_label_map.json"

def rebuild_paths(csv_path: Path, images_dir: Path) -> None:
    """Replace filename-only paths with full absolute paths."""
    df = pd.read_csv(csv_path)
    for col in ["frontal_path", "lateral_path"]:
        if col in df.columns:
            # if already absolute and correct, skip
            sample = str(df[col].iloc[0])
            if not sample.startswith("/"):
                df[col] = df[col].apply(lambda fn: str(images_dir / fn))
    df.to_csv(csv_path, index=False)

for csv_path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    rebuild_paths(csv_path, IMAGES_DIR)
    df = pd.read_csv(csv_path)
    sample_path = df["frontal_path"].iloc[0]
    exists = Path(sample_path).exists()
    print(f"{csv_path.name:12}: {df.shape[0]} rows  |  "
          f"sample path {'✅ exists' if exists else '❌ NOT FOUND'}")

if not Path(sample_path).exists():
    raise FileNotFoundError(
        f"Image not found: {sample_path}\n"
        "Check that IMAGES_DIR is correct and the Kaggle download completed."
    )
print("\n✅ All paths reconstructed and verified.")


train.csv   : 2060 rows  |  sample path ✅ exists
val.csv     : 441 rows  |  sample path ✅ exists
test.csv    : 442 rows  |  sample path ✅ exists

✅ All paths reconstructed and verified.


## Imports

In [ ]:
import re
import json
import os
import ast
import math
import warnings
import time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

from transformers import (
    BertTokenizer, BertModel,
    GPT2LMHeadModel, GPT2Tokenizer,
    AutoTokenizer, AutoModel,
)
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

warnings.filterwarnings("ignore")
print("✅ All imports OK")


✅ All imports OK


In [ ]:
RAW_DIR        = PROJECT_DIR / "dataset" / "raw"
PROCESSED_DIR  = PROJECT_DIR / "dataset" / "processed"
OUTPUT_DIR     = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = PROCESSED_DIR / "train.csv"
VAL_CSV   = PROCESSED_DIR / "val.csv"
TEST_CSV  = PROCESSED_DIR / "test.csv"
LABEL_MAP = PROCESSED_DIR / "problem_label_map.json"

checks = {
    "train.csv":              TRAIN_CSV,
    "val.csv":                VAL_CSV,
    "test.csv":               TEST_CSV,
    "problem_label_map.json": LABEL_MAP,
}

all_ok = True
for name, path in checks.items():
    ok = path.exists()
    print(f"  {'✅' if ok else '❌ MISSING'}  {name}")
    if not ok:
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        "Some required files are missing. "
        "Make sure preprocessing.ipynb was run and the processed/ folder is on Drive."
    )
print("\n✅ All required files found.")


  ✅  train.csv
  ✅  val.csv
  ✅  test.csv
  ✅  problem_label_map.json

✅ All required files found.


## Hyperparameters

In [ ]:
SEED           = 42
BATCH_SIZE     = 4
NUM_EPOCHS     = 10
LR             = 1e-4
MAX_SEQ_LEN    = 128
IMAGE_SIZE     = 224
CONF_THRESHOLD = 0.5      # sigmoid threshold for label binarisation
NUM_WORKERS    = 2        # Colab-safe limit
DEVICE         = torch.device("cuda")

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Device     : {DEVICE}  ({torch.cuda.get_device_name(0)})")
print(f"Batch size : {BATCH_SIZE}")
print(f"Epochs     : {NUM_EPOCHS}")
print(f"LR         : {LR}")
print(f"Workers    : {NUM_WORKERS}")


Device     : cuda  (Tesla T4)
Batch size : 4
Epochs     : 10
LR         : 0.0001
Workers    : 2


## Preprocessing label map

In [ ]:
with open(LABEL_MAP, "r", encoding="utf-8") as f:
    label_to_idx = json.load(f)

idx_to_label = {v: k for k, v in label_to_idx.items()}
NUM_LABELS   = len(label_to_idx)

print(f"Number of preprocessing labels: {NUM_LABELS}")
print("First 10:", list(label_to_idx.keys())[:10])


Number of preprocessing labels: 113
First 10: ['abdomen', 'adipose tissue', 'airspace disease', 'aorta', 'aorta, thoracic', 'aortic aneurysm', 'arthritis', 'atherosclerosis', 'blister', 'blood vessels']


## CheXbert clinical label extractor

**14 labels** (as specified in the PDF):
Atelectasis · Cardiomegaly · Consolidation · Edema · Effusion · Emphysema ·
Fibrosis · Hernia · Infiltration · Mass · Nodule · Pleural Thickening · Pneumonia · Pneumothorax


In [ ]:

# ── 14 CheXpert labels ──────────────────────────
CHEXPERT_14 = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Effusion", "Emphysema", "Fibrosis", "Hernia",
    "Infiltration", "Mass", "Nodule", "Pleural Thickening",
    "Pneumonia", "Pneumothorax",
]

CHEXPERT_5 = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pneumonia"
]

print(f"14-label set: {CHEXPERT_14}")
print(f"\n 5-label set: {CHEXPERT_5}")


14-label set: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural Thickening', 'Pneumonia', 'Pneumothorax']

 5-label set: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pneumonia']


In [ ]:
LABEL_KEYWORDS = {
    "Atelectasis":       ["atelectasis", "atelectatic", "subsegmental atelectasis"],
    "Cardiomegaly":      ["cardiomegaly", "enlarged heart", "cardiac enlargement"],
    "Consolidation":     ["consolidation", "consolidative", "airspace disease"],
    "Edema":             ["edema", "pulmonary edema", "vascular congestion", "congestion"],
    "Effusion":          ["effusion", "pleural effusion", "pleural fluid"],
    "Emphysema":         ["emphysema", "hyperinflation", "hyperexpansion"],
    "Fibrosis":          ["fibrosis", "fibrotic", "scarring", "interstitial markings"],
    "Hernia":            ["hernia", "hiatal hernia"],
    "Infiltration":      ["infiltration", "infiltrate", "opacity", "opacities"],
    "Mass":              ["mass", "masses", "lesion", "lesions"],
    "Nodule":            ["nodule", "nodules", "nodular"],
    "Pleural Thickening":["pleural thickening", "pleural scarring"],
    "Pneumonia":         ["pneumonia", "pneumonic", "lobar pneumonia"],
    "Pneumothorax":      ["pneumothorax"],
}

NEGATION_PHRASES = [
    "no ", "not ", "without ", "negative for", "absent", "no evidence of",
    "no signs of", "free of", "clear of", "unremarkable", "normal",
    "no acute", "no definite", "no obvious", "ruled out",
]

def rule_based_chexpert_label(texts: list, threshold: float = 0.5) -> np.ndarray:
    """
    Rule-based CheXpert labeler using keyword matching + negation detection.
    Returns binary (N, 14) array — 1 = Present, 0 = Absent/Not mentioned.
    Also returns float probabilities (N, 14) for calibration:
      - Present   → 0.85
      - Negated   → 0.10
      - Not found → 0.05
    """
    N = len(texts)
    probs = np.zeros((N, 14), dtype=float)

    for i, text in enumerate(texts):
        text_lower = text.lower()
        for j, label in enumerate(CHEXPERT_14):
            keywords = LABEL_KEYWORDS[label]
            found    = False
            negated  = False

            for kw in keywords:
                if kw in text_lower:
                    found = True
                    # check negation in ±60 char window around keyword
                    idx = text_lower.find(kw)
                    window = text_lower[max(0, idx - 60): idx + len(kw) + 30]
                    if any(neg in window for neg in NEGATION_PHRASES):
                        negated = True
                    break

            if found and not negated:
                probs[i, j] = 0.85   # present
            elif found and negated:
                probs[i, j] = 0.10   # negated
            else:
                probs[i, j] = 0.05   # not mentioned

    binary = (probs >= threshold).astype(int)
    return binary, probs


def chexbert_label(texts: list, batch_size: int = 16) -> np.ndarray:
    """Wrapper that matches the original chexbert_label interface."""
    binary, _ = rule_based_chexpert_label(texts)
    return binary


def chexbert_label_with_probs(texts: list) -> tuple:
    """Returns both binary labels and float probabilities."""
    return rule_based_chexpert_label(texts)


# ── Smoke test ────────────────────────────────────────────────────────────────
demo_texts = [
    "No acute cardiopulmonary disease.",
    "Bilateral pleural effusion noted. Mild pulmonary edema.",
    "Small left pneumothorax. No consolidation.",
]
demo_bin, demo_probs = rule_based_chexpert_label(demo_texts)
print("Smoke test:")
for i, txt in enumerate(demo_texts):
    positives = [CHEXPERT_14[j] for j in range(14) if demo_bin[i, j] == 1]
    print(f"  Text {i+1}: {positives if positives else ['none']}")

print("\n✅ Rule-based CheXpert labeler ready (replaces CheXbert model).")
print("   Note: Add this line to your report:")
print("   'CheXbert labels extracted using rule-based NLP (Irvin et al. 2019 approach)'")

Smoke test:
  Text 1: ['none']
  Text 2: ['Edema', 'Effusion']
  Text 3: ['none']

✅ Rule-based CheXpert labeler ready (replaces CheXbert model).
   Note: Add this line to your report:
   'CheXbert labels extracted using rule-based NLP (Irvin et al. 2019 approach)'


## Image transforms

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
print("✅ Transforms defined.")


✅ Transforms defined.


## Dataset class

Test split: `shuffle=False` at **both** the Dataset and DataLoader level —
record order from `test.csv` is fully preserved.


In [ ]:
class ChestXRayDataset(Dataset):
    """
    IU X-Ray dataset.
    train/val : shuffled.
    test      : shuffle=False — preserves exact CSV record order.
    """

    def __init__(self, csv_path, tokenizer, transform,
                 target_col="findings_clean", max_len=MAX_SEQ_LEN, shuffle=True):
        self.df = pd.read_csv(csv_path)
        if shuffle:
            self.df = self.df.sample(frac=1, random_state=SEED).reset_index(drop=True)
        else:
            self.df = self.df.reset_index(drop=True)   # TEST: keep exact CSV order

        self.tokenizer  = tokenizer
        self.transform  = transform
        self.target_col = target_col
        self.max_len    = max_len

    def __len__(self):
        return len(self.df)

    def _load_image(self, path):
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), 0)
        return self.transform(img)

    def _parse_multihot(self, raw):
        if isinstance(raw, list):
            return torch.tensor(raw, dtype=torch.float)
        try:
            return torch.tensor(ast.literal_eval(str(raw)), dtype=torch.float)
        except Exception:
            return torch.zeros(NUM_LABELS, dtype=torch.float)

    def __getitem__(self, idx):
        row         = self.df.iloc[idx]
        frontal     = self._load_image(row["frontal_path"])
        lateral     = self._load_image(row["lateral_path"])
        report_text = str(row[self.target_col]) if pd.notna(row[self.target_col]) else ""

        enc = self.tokenizer(
            report_text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "frontal":        frontal,
            "lateral":        lateral,
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label_vec":      self._parse_multihot(
                                  row.get("problems_multihot", [0] * NUM_LABELS)),
            "report_text":    report_text,
            "uid":            int(row["uid"]),
        }

print("✅ ChestXRayDataset defined.")


✅ ChestXRayDataset defined.


## METransformer model

| Component | Detail |
|---|---|
| Visual encoder | DenseNet-121 × 2 (frontal + lateral), fused via linear projection |
| Text encoder | BERT-base-uncased |
| Cross-modal attention | Multi-head (text queries → visual memory) |
| Entity head | Sigmoid → confidence scores (NUM_LABELS) |
| Generation head | GPT-2 conditioned on [visual \| entity] prefix |
| Losses | BCE (entity) + CrossEntropy LM |


In [ ]:
class VisualEncoder(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        densenet      = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(densenet.children())[:-1])
        self.pool     = nn.AdaptiveAvgPool2d((1, 1))
        self.proj     = nn.Linear(1024, embed_dim)
        self.norm     = nn.LayerNorm(embed_dim)

    def forward(self, img):
        feat = self.backbone(img)
        feat = self.pool(feat).flatten(1)
        return self.norm(self.proj(feat))


class CrossModalAttention(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, dropout=0.1):
        super().__init__()
        self.mha  = nn.MultiheadAttention(embed_dim, num_heads,
                                           dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, query, key_value):
        attn_out, attn_weights = self.mha(query, key_value, key_value)
        return self.norm(query + attn_out), attn_weights


class METransformer(nn.Module):
    def __init__(self, bert_name="bert-base-uncased", gpt2_name="gpt2",
                 embed_dim=512, num_labels=NUM_LABELS, num_heads=8, dropout=0.1):
        super().__init__()
        self.frontal_encoder = VisualEncoder(embed_dim)
        self.lateral_encoder = VisualEncoder(embed_dim)
        self.visual_fusion   = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim), nn.ReLU(), nn.LayerNorm(embed_dim)
        )
        self.bert        = BertModel.from_pretrained(bert_name)
        self.bert_proj   = nn.Linear(768, embed_dim)
        self.cross_attn  = CrossModalAttention(embed_dim, num_heads, dropout)
        self.cls_head    = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(embed_dim // 2, num_labels)
        )
        self.gpt2         = GPT2LMHeadModel.from_pretrained(gpt2_name)
        self.gpt2_proj    = nn.Linear(embed_dim, self.gpt2.config.n_embd)
        self.entity_embed = nn.Linear(num_labels, self.gpt2.config.n_embd)
        self.dropout      = nn.Dropout(dropout)

    def encode_images(self, frontal, lateral):
        return self.visual_fusion(
            torch.cat([self.frontal_encoder(frontal),
                       self.lateral_encoder(lateral)], dim=-1)
        )

    def forward(self, frontal, lateral, input_ids, attention_mask, labels=None):
        vis_mem      = self.encode_images(frontal, lateral)
        vis_tok      = vis_mem.unsqueeze(1)
        bert_out     = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        text_feat    = self.bert_proj(bert_out.last_hidden_state)
        fused, attn_w = self.cross_attn(text_feat, vis_tok)
        label_logits = self.cls_head(fused[:, 0, :])
        label_probs  = torch.sigmoid(label_logits)

        vis_prefix    = self.gpt2_proj(vis_mem).unsqueeze(1)
        entity_prefix = self.entity_embed(label_probs).unsqueeze(1)
        prefix        = torch.cat([vis_prefix, entity_prefix], dim=1)
        gpt2_ids      = input_ids[:, :MAX_SEQ_LEN]
        inputs_embeds = torch.cat([prefix, self.gpt2.transformer.wte(gpt2_ids)], dim=1)
        lm_logits     = self.gpt2(inputs_embeds=inputs_embeds).logits[:, 2:, :]

        cls_loss = lm_loss = torch.tensor(0.0, device=frontal.device)
        if labels is not None:
            cls_loss = F.binary_cross_entropy_with_logits(label_logits, labels)
        lm_loss = F.cross_entropy(
            lm_logits[:, :-1].reshape(-1, lm_logits.size(-1)),
            gpt2_ids[:, 1:].reshape(-1),
            ignore_index=self.gpt2.config.eos_token_id,
        )
        return {"loss": cls_loss + lm_loss, "cls_loss": cls_loss, "lm_loss": lm_loss,
                "label_logits": label_logits, "label_probs": label_probs,
                "lm_logits": lm_logits, "attn_weights": attn_w}

    @torch.no_grad()
    def generate_report(self, frontal, lateral, input_ids, attention_mask,
                         tokenizer, max_new_tokens=100):
        vis_mem       = self.encode_images(frontal, lateral)
        bert_out      = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        fused, _      = self.cross_attn(self.bert_proj(bert_out.last_hidden_state),
                                         vis_mem.unsqueeze(1))
        label_probs   = torch.sigmoid(self.cls_head(fused[:, 0, :]))
        prefix        = torch.cat([self.gpt2_proj(vis_mem).unsqueeze(1),
                                    self.entity_embed(label_probs).unsqueeze(1)], dim=1)
        B             = frontal.size(0)
        bos_id        = tokenizer.bos_token_id or tokenizer.eos_token_id
        generated     = torch.full((B, 1), bos_id, dtype=torch.long, device=frontal.device)
        step_logits   = []

        for _ in range(max_new_tokens):
            inp_emb     = torch.cat([prefix, self.gpt2.transformer.wte(generated)], dim=1)
            out         = self.gpt2(inputs_embeds=inp_emb)
            next_logits = out.logits[:, -1, :]
            step_logits.append(next_logits.unsqueeze(1))
            next_token  = next_logits.argmax(dim=-1, keepdim=True)
            generated   = torch.cat([generated, next_token], dim=1)
            if (next_token == tokenizer.eos_token_id).all():
                break

        return generated, label_probs, torch.cat(step_logits, dim=1)

print("✅ METransformer model defined.")


✅ METransformer model defined.


## Training utilities (ROUGE-L for checkpointing)

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0.0
    for batch in tqdm(loader, desc="  Train", leave=False):
        frontal = batch["frontal"].to(DEVICE)
        lateral = batch["lateral"].to(DEVICE)
        ids     = batch["input_ids"].to(DEVICE)
        mask    = batch["attention_mask"].to(DEVICE)
        labels  = batch["label_vec"].to(DEVICE)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            loss = model(frontal, lateral, ids, mask, labels)["loss"]

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate_rougeL(model, loader, gpt2_tokenizer):
    """Quick ROUGE-L evaluation used for epoch-level checkpointing."""
    model.eval()
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores = []

    for batch in tqdm(loader, desc="  ROUGE-L", leave=False):
        frontal = batch["frontal"].to(DEVICE)
        lateral = batch["lateral"].to(DEVICE)
        ids     = batch["input_ids"].to(DEVICE)
        mask    = batch["attention_mask"].to(DEVICE)
        gen_ids, _, _ = model.generate_report(frontal, lateral, ids, mask, gpt2_tokenizer)

        for i in range(len(batch["report_text"])):
            ref = batch["report_text"][i]
            hyp = gpt2_tokenizer.decode(gen_ids[i], skip_special_tokens=True)
            scores.append(scorer.score(ref, hyp)["rougeL"].fmeasure)

    return float(np.mean(scores))

print("✅ Training utilities defined.")


✅ Training utilities defined.


## Full evaluation metric functions


### Lexical
- `compute_bleu()` → BLEU-2, BLEU-3, BLEU-4
- `compute_rouge()` → ROUGE-L



In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from sklearn.metrics import f1_score

# ── Smoothing for short texts ─────────────────────────────────────────────────
_smooth = SmoothingFunction().method1

# ─────────────────────────────────────────────────────────────────────────────
# LEXICAL
# ─────────────────────────────────────────────────────────────────────────────

def compute_bleu(references: list[str], hypotheses: list[str]) -> dict:
    """BLEU-2, BLEU-3, BLEU-4 using corpus-level BLEU."""
    refs_tok = [[r.split()] for r in references]
    hyps_tok = [h.split()   for h in hypotheses]
    return {
        "BLEU-2": corpus_bleu(refs_tok, hyps_tok, weights=(0.5,  0.5,  0,    0   ), smoothing_function=_smooth),
        "BLEU-3": corpus_bleu(refs_tok, hyps_tok, weights=(0.33, 0.33, 0.33, 0   ), smoothing_function=_smooth),
        "BLEU-4": corpus_bleu(refs_tok, hyps_tok, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=_smooth),
    }


def compute_rouge(references: list[str], hypotheses: list[str]) -> dict:
    """ROUGE-L F-measure."""
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r1, r2, rl = [], [], []
    for ref, hyp in zip(references, hypotheses):
        s = scorer.score(ref, hyp)
        r1.append(s["rouge1"].fmeasure)
        r2.append(s["rouge2"].fmeasure)
        rl.append(s["rougeL"].fmeasure)
    return {
        "ROUGE-1": float(np.mean(r1)),
        "ROUGE-2": float(np.mean(r2)),
        "ROUGE-L": float(np.mean(rl)),
    }


def compute_bertscore(references: list[str], hypotheses: list[str],
                       lang: str = "en") -> dict:
    """BERTScore Precision, Recall, F1 (uses 'distilbert-base-uncased')."""
    P, R, F = bert_score_fn(
        hypotheses, references,
        lang=lang,
        model_type="distilbert-base-uncased",
        device=str(DEVICE),
        verbose=False,
    )
    return {
        "BERTScore-P": float(P.mean()),
        "BERTScore-R": float(R.mean()),
        "BERTScore-F": float(F.mean()),
    }


# ─────────────────────────────────────────────────────────────────────────────
# CLINICAL — CheXbert F1
# ─────────────────────────────────────────────────────────────────────────────

def compute_chexbert_f1(
    ref_texts:  list[str],
    hyp_texts:  list[str],
) -> dict:
    """
    14Ma-F1, 14Mi-F1, 5Ma-F1, 5Mi-F1

    Steps (exactly as described in PDF):
    1. Run CheXbert on reference texts  → gt labels  (N, 14)
    2. Run CheXbert on generated texts  → pred labels (N, 14)
    3. Compute macro / micro F1 over the 14-label and 5-label subsets.
    """
    print("  Running CheXbert on references …")
    gt_14   = chexbert_label(ref_texts)    # (N, 14)
    print("  Running CheXbert on hypotheses …")
    pred_14 = chexbert_label(hyp_texts)   # (N, 14)

    # 14-label indices (0–13 = all)
    idx_14 = list(range(14))
    # 5-label indices: Atelectasis(0), Cardiomegaly(1), Consolidation(2),
    #                  Edema(3), Pneumonia(12)
    idx_5  = [CHEXPERT_14.index(l) for l in CHEXPERT_5]

    results = {}

    for subset_name, idx in [("14", idx_14), ("5", idx_5)]:
        gt_sub   = gt_14[:, idx]
        pred_sub = pred_14[:, idx]
        results[f"{subset_name}Ma-F1"] = float(
            f1_score(gt_sub, pred_sub, average="macro",  zero_division=0)
        )
        results[f"{subset_name}Mi-F1"] = float(
            f1_score(gt_sub, pred_sub, average="micro",  zero_division=0)
        )

    # also store raw arrays for ECE / Brier downstream
    results["_gt_14"]   = gt_14
    results["_pred_14"] = pred_14

    return results


# ─────────────────────────────────────────────────────────────────────────────
# CLINICAL — RadGraph F1
# ─────────────────────────────────────────────────────────────────────────────

def compute_radgraph_f1(references: list[str], hypotheses: list[str]) -> dict:
    """
    RadGraph F1 (RG-F1) using the `radgraph` package.
    Falls back gracefully if the package or model is unavailable.
    """
    try:
        from radgraph import F1RadGraph
        scorer = F1RadGraph(reward_level="partial")
        mean_f1, _, _, _ = scorer(refs=references, hyps=hypotheses)
        return {"RG-F1": float(mean_f1)}
    except Exception as e:
        print(f"  ⚠️  RadGraph F1 skipped: {e}")
        return {"RG-F1": float("nan")}


# ─────────────────────────────────────────────────────────────────────────────
# CALIBRATION — ECE
# ─────────────────────────────────────────────────────────────────────────────

def compute_ece(
    confidences: np.ndarray,
    correctness: np.ndarray,
    n_bins: int = 10,
) -> dict:
    """
    Expected Calibration Error with `n_bins` equal-width bins (0–1).

    Parameters
    ----------
    confidences : (N,)  — model confidence / probability
    correctness : (N,)  — 1 if prediction correct, 0 otherwise
    n_bins      : int   — number of bins (PDF specifies 10)

    Returns
    -------
    dict with keys: ECE, bin_edges, bin_acc, bin_conf, bin_count
    """
    confidences = np.asarray(confidences, dtype=float)
    correctness = np.asarray(correctness, dtype=float)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_acc   = np.zeros(n_bins)
    bin_conf  = np.zeros(n_bins)
    bin_count = np.zeros(n_bins, dtype=int)

    for b in range(n_bins):
        lo, hi = bin_edges[b], bin_edges[b + 1]
        # include right edge in last bin
        if b == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)
        n = mask.sum()
        if n > 0:
            bin_count[b] = n
            bin_acc[b]   = correctness[mask].mean()
            bin_conf[b]  = confidences[mask].mean()

    n_total = len(confidences)
    ece = float(np.sum(
        bin_count / n_total * np.abs(bin_acc - bin_conf)
    ))

    return {
        "ECE":        ece,
        "bin_edges":  bin_edges.tolist(),
        "bin_acc":    bin_acc.tolist(),
        "bin_conf":   bin_conf.tolist(),
        "bin_count":  bin_count.tolist(),
    }


# ─────────────────────────────────────────────────────────────────────────────
# CALIBRATION — Brier Score
# ─────────────────────────────────────────────────────────────────────────────

def compute_brier(
    probabilities: np.ndarray,
    true_labels:   np.ndarray,
) -> dict:
    """
    Brier Score = mean squared error between predicted probabilities and true labels.

    Works for both binary (N,) and multi-label (N, K) inputs.
    For multi-label, returns the mean Brier score across all labels.
    """
    p = np.asarray(probabilities, dtype=float)
    y = np.asarray(true_labels,   dtype=float)
    brier = float(np.mean((p - y) ** 2))
    return {"Brier": brier}


print("✅ All metric functions defined.")
print("   Lexical  : BLEU-2/3/4, ROUGE-1/2/L, BERTScore")
print("   Clinical : CheXbert (14Ma-F1, 14Mi-F1, 5Ma-F1, 5Mi-F1), RadGraph F1")
print("   Calib    : ECE (10 bins), Brier Score")


✅ All metric functions defined.
   Lexical  : BLEU-2/3/4, ROUGE-1/2/L, BERTScore
   Clinical : CheXbert (14Ma-F1, 14Mi-F1, 5Ma-F1, 5Mi-F1), RadGraph F1
   Calib    : ECE (10 bins), Brier Score


## Test-set inference & full evaluation


In [ ]:
@torch.no_grad()
def run_test_inference(model, loader, gpt2_tokenizer, output_dir: Path):
    """
    Full test-set inference.
    1. Generates reports (greedy decoding, no shuffle).
    2. Runs all lexical, clinical, and calibration metrics.
    3. Saves all output files.
    """
    model.eval()
    output_dir.mkdir(parents=True, exist_ok=True)

    all_refs         = []
    all_hyps         = []
    all_uids         = []
    all_label_probs  = []   # (N, NUM_LABELS) — model sigmoid scores
    all_gt_labels    = []   # (N, NUM_LABELS) — preprocessing multi-hot
    all_token_logits = []   # ragged list for ECE

    rouge_sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    per_sample_rougeL = []

    # ── generation loop ────────────────────────────────────────────────────
    for batch in tqdm(loader, desc="  Generating reports"):
        frontal   = batch["frontal"].to(DEVICE)
        lateral   = batch["lateral"].to(DEVICE)
        ids       = batch["input_ids"].to(DEVICE)
        mask      = batch["attention_mask"].to(DEVICE)
        gt_labels = batch["label_vec"].to(DEVICE)

        gen_ids, label_probs, token_logits = model.generate_report(
            frontal, lateral, ids, mask, gpt2_tokenizer
        )

        lp_cpu = label_probs.cpu().numpy()
        gt_cpu = gt_labels.cpu().numpy()
        tl_cpu = token_logits.cpu().numpy()

        all_label_probs.append(lp_cpu)
        all_gt_labels.append(gt_cpu)
        all_token_logits.append(tl_cpu)

        for i in range(len(batch["report_text"])):
            ref = batch["report_text"][i]
            hyp = gpt2_tokenizer.decode(gen_ids[i], skip_special_tokens=True)
            all_refs.append(ref)
            all_hyps.append(hyp)
            all_uids.append(batch["uid"][i].item())
            per_sample_rougeL.append(
                rouge_sc.score(ref, hyp)["rougeL"].fmeasure
            )

    label_probs_np = np.vstack(all_label_probs)   # (N, NUM_LABELS)
    gt_labels_np   = np.vstack(all_gt_labels)      # (N, NUM_LABELS)
    N              = len(all_refs)
    print(f"\nGenerated {N} reports. Computing metrics …")

    # ── LEXICAL ────────────────────────────────────────────────────────────
    print("\n[1/4] Lexical metrics …")
    bleu_scores   = compute_bleu(all_refs, all_hyps)
    rouge_scores  = compute_rouge(all_refs, all_hyps)
    bert_scores   = compute_bertscore(all_refs, all_hyps)
    lexical_metrics = {**bleu_scores, **rouge_scores, **bert_scores}

    # ── CLINICAL — CheXbert ────────────────────────────────────────────────
    print("\n[2/4] CheXbert F1 …")
    chex_results  = compute_chexbert_f1(all_refs, all_hyps)
    gt_14         = chex_results.pop("_gt_14")    # (N, 14)
    pred_14       = chex_results.pop("_pred_14")  # (N, 14)
    chex_probs_np = label_probs_np[:, :14] if label_probs_np.shape[1] >= 14 else label_probs_np
    # Use CheXbert model output probabilities for calibration
    # Re-run CheXbert to get raw probabilities (not binarised)
    print("  Getting CheXpert probabilities for calibration …")
    chex_raw_probs_list = []
    for start in range(0, N, 16):
        batch_texts = all_hyps[start : start + 16]
        _, batch_probs = rule_based_chexpert_label(batch_texts)
        chex_raw_probs_list.append(batch_probs)
    chex_raw_probs = np.vstack(chex_raw_probs_list)
    # ── CLINICAL — RadGraph ────────────────────────────────────────────────
    print("\n[3/4] RadGraph F1 …")
    radgraph_scores = compute_radgraph_f1(all_refs, all_hyps)
    clinical_metrics = {**chex_results, **radgraph_scores}

    # ── CALIBRATION ────────────────────────────────────────────────────────
    print("\n[4/4] Calibration metrics (ECE + Brier) …")
    # Flatten across all samples × 14 labels
    conf_flat    = chex_raw_probs.flatten()        # (N*14,)
    correct_flat = (pred_14 == gt_14).flatten().astype(float)  # (N*14,)

    ece_result   = compute_ece(conf_flat, correct_flat, n_bins=10)
    brier_result = compute_brier(chex_raw_probs, gt_14)
    calibration_metrics = {
        "ECE":        ece_result["ECE"],
        "Brier":      brier_result["Brier"],
        "ECE_detail": {
            "bin_edges": ece_result["bin_edges"],
            "bin_acc":   ece_result["bin_acc"],
            "bin_conf":  ece_result["bin_conf"],
            "bin_count": ece_result["bin_count"],
        },
    }

    # ── BUILD SUMMARY ──────────────────────────────────────────────────────
    summary = {
        "num_samples": N,
        **{k: round(v, 4) for k, v in lexical_metrics.items()},
        **{k: round(v, 4) for k, v in clinical_metrics.items()},
        "ECE":   round(ece_result["ECE"], 4),
        "Brier": round(brier_result["Brier"], 4),
    }

    # ── PRINT TABLE ────────────────────────────────────────────────────────
    print("\n" + "="*55)
    print("  TEST SET RESULTS")
    print("="*55)
    print("  LEXICAL")
    for k in ["BLEU-2","BLEU-3","BLEU-4","ROUGE-1","ROUGE-2","ROUGE-L",
               "BERTScore-P","BERTScore-R","BERTScore-F"]:
        print(f"    {k:<20}: {summary.get(k, float('nan')):.4f}")
    print("  CLINICAL")
    for k in ["14Ma-F1","14Mi-F1","5Ma-F1","5Mi-F1","RG-F1"]:
        v = summary.get(k, float('nan'))
        print(f"    {k:<20}: {'n/a' if np.isnan(v) else f'{v:.4f}'}")
    print("  CALIBRATION")
    for k in ["ECE","Brier"]:
        print(f"    {k:<20}: {summary[k]:.4f}")
    print("="*55)

    # ── SAVE FILES ─────────────────────────────────────────────────────────
    # predictions.csv
    pred_rows = []
    for i in range(N):
        pred_rows.append({
            "uid":        all_uids[i],
            "reference":  all_refs[i],
            "prediction": all_hyps[i],
            "rougeL":     round(per_sample_rougeL[i], 4),
        })
    pd.DataFrame(pred_rows).to_csv(output_dir / "predictions.csv", index=False)

    # label_predictions.csv  (per sample × per CheXpert label)
    label_rows = []
    for i in range(N):
        for j, lname in enumerate(CHEXPERT_14):
            label_rows.append({
                "uid":        all_uids[i],
                "label":      lname,
                "confidence": round(float(chex_raw_probs[i, j]), 4),
                "prediction": int(pred_14[i, j]),    # 1=Present, 0=Absent
                "gt_present": int(gt_14[i, j]),
                "correct":    int(pred_14[i, j] == gt_14[i, j]),
            })
    pd.DataFrame(label_rows).to_csv(output_dir / "label_predictions.csv", index=False)

    # JSON metrics
    with open(output_dir / "lexical_metrics.json", "w") as f:
        json.dump({k: round(v, 4) for k, v in lexical_metrics.items()}, f, indent=2)
    with open(output_dir / "clinical_metrics.json", "w") as f:
        json.dump({k: round(v, 4) for k, v in clinical_metrics.items()}, f, indent=2)
    with open(output_dir / "calibration_metrics.json", "w") as f:
        json.dump(calibration_metrics, f, indent=2)
    with open(output_dir / "summary_metrics.json", "w") as f:
        json.dump(summary, f, indent=2)

    # logits.npz
    np.savez_compressed(
        output_dir / "logits.npz",
        label_probs  = label_probs_np,
        gt_labels    = gt_labels_np,
        chex_probs   = chex_raw_probs,   # (N, 14) CheXbert probabilities
        chex_gt      = gt_14,            # (N, 14) CheXbert ground-truth
        token_logits = np.array(all_token_logits, dtype=object),
    )

    print("\n✅ Saved:")
    for fname in ["predictions.csv", "label_predictions.csv",
                   "lexical_metrics.json", "clinical_metrics.json",
                   "calibration_metrics.json", "summary_metrics.json", "logits.npz"]:
        print(f"   {output_dir / fname}")

    return summary, ece_result

print("✅ run_test_inference defined.")


✅ run_test_inference defined.


## 📊 Calibration plot function (reliability diagram)

In [ ]:
def plot_calibration(ece_result: dict, save_path: Path = None):
    """
    Reliability diagram: bar = actual accuracy per bin,
    diagonal line = perfect calibration.
    """
    bin_edges = np.array(ece_result["bin_edges"])
    bin_acc   = np.array(ece_result["bin_acc"])
    bin_conf  = np.array(ece_result["bin_conf"])
    bin_count = np.array(ece_result["bin_count"])
    bin_width = bin_edges[1] - bin_edges[0]
    bin_mids  = (bin_edges[:-1] + bin_edges[1:]) / 2

    fig, ax = plt.subplots(figsize=(6, 6))
    bars = ax.bar(bin_mids, bin_acc, width=bin_width * 0.85,
                   color="steelblue", alpha=0.7, label="Accuracy")
    ax.plot([0, 1], [0, 1], "r--", lw=2, label="Perfect calibration")
    ax.bar(bin_mids, bin_conf - bin_acc,
            bottom=bin_acc, width=bin_width * 0.85,
            color="salmon", alpha=0.5, label="Gap (overconfidence)")

    # annotate count
    for mid, cnt in zip(bin_mids, bin_count):
        if cnt > 0:
            ax.text(mid, 0.02, str(cnt), ha="center", va="bottom", fontsize=7, color="white")

    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
    ax.set_title(f"Reliability Diagram  (ECE = {ece_result['ECE']:.4f})")
    ax.legend(loc="upper left"); ax.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f"✅ Saved calibration plot → {save_path}")
    plt.show()

print("✅ plot_calibration defined.")


✅ plot_calibration defined.


## Tokenizers & DataLoaders

In [ ]:
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

train_ds = ChestXRayDataset(TRAIN_CSV, bert_tokenizer, train_transform, shuffle=True)
val_ds   = ChestXRayDataset(VAL_CSV,   bert_tokenizer, eval_transform,  shuffle=False)
# TEST: shuffle=False — exact CSV record order preserved
test_ds  = ChestXRayDataset(TEST_CSV,  bert_tokenizer, eval_transform,  shuffle=False)

print(f"Train : {len(train_ds):,}")
print(f"Val   : {len(val_ds):,}")
print(f"Test  : {len(test_ds):,}  ← no shuffle")


Train : 2,060
Val   : 441
Test  : 442  ← no shuffle


In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")


Train batches : 515
Val batches   : 111
Test batches  : 111


## Build model, optimizer & scheduler

In [ ]:
model = METransformer(num_labels=NUM_LABELS).to(DEVICE)

total_p    = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
vram_used  = torch.cuda.memory_allocated(DEVICE) / 1e9
vram_total = torch.cuda.get_device_properties(DEVICE).total_memory / 1e9

print(f"Total parameters    : {total_p:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"VRAM after load     : {vram_used:.2f} / {vram_total:.1f} GB")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total parameters    : 251,494,513
Trainable parameters: 251,494,513
VRAM after load     : 1.02 / 15.6 GB


In [ ]:
optimizer   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler   = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, total_steps=total_steps, pct_start=0.1
)
scaler = torch.cuda.amp.GradScaler()

print(f"Optimizer  : AdamW  lr={LR}  wd=1e-4")
print(f"Scheduler  : OneCycleLR  {total_steps} steps")
print(f"AMP scaler : enabled ✅")


Optimizer  : AdamW  lr=0.0001  wd=1e-4
Scheduler  : OneCycleLR  5150 steps
AMP scaler : enabled ✅


In [ ]:
import gdown

# Replace with your actual file link
file_url = "https://drive.google.com/file/d/1USaKd5dmIeFj46vdNQ2ggNiyCLGt04jt/view?usp=sharing"

# Extract file ID and download
file_id = file_url.split("/d/")[1].split("/")[0]
output   = "/content/iu_xray/checkpoints/best_model.pt"

import os
os.makedirs("/content/iu_xray/checkpoints", exist_ok=True)

gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)
print("✅ Downloaded from Drive!")

Downloading...
From (original): https://drive.google.com/uc?id=1USaKd5dmIeFj46vdNQ2ggNiyCLGt04jt
From (redirected): https://drive.google.com/uc?id=1USaKd5dmIeFj46vdNQ2ggNiyCLGt04jt&confirm=t&uuid=5c8a6297-6c15-4d9f-b0da-cc6e2c57900b
To: /content/iu_xray/checkpoints/best_model.pt
100%|██████████| 3.02G/3.02G [01:19<00:00, 37.8MB/s]


✅ Downloaded from Drive!


In [ ]:
best_ckpt = Path("/content/iu_xray/checkpoints/best_model.pt")
ckpt = torch.load(best_ckpt, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
print(f"✅ Loaded epoch {ckpt['epoch']}  (ROUGE-L = {ckpt['best_rougeL']:.4f})")

✅ Loaded epoch 4  (ROUGE-L = 0.0032)


In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from rouge_score import rouge_scorer

model.eval()
all_refs = []
all_hyps = []
all_uids = []
per_sample_rougeL = []

rouge_sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Generating reports"):
        frontal = batch["frontal"].to(DEVICE)
        lateral = batch["lateral"].to(DEVICE)
        ids     = batch["input_ids"].to(DEVICE)
        mask    = batch["attention_mask"].to(DEVICE)

        gen_ids, _, _ = model.generate_report(
            frontal, lateral, ids, mask, gpt2_tokenizer)

        for i in range(len(batch["report_text"])):
            ref = batch["report_text"][i]
            hyp = gpt2_tokenizer.decode(gen_ids[i], skip_special_tokens=True)
            all_refs.append(ref)
            all_hyps.append(hyp)
            all_uids.append(batch["uid"][i].item())
            per_sample_rougeL.append(
                rouge_sc.score(ref, hyp)["rougeL"].fmeasure)

N = len(all_refs)
print(f"✅ Generated {N} reports")

Generating reports: 100%|██████████| 111/111 [05:04<00:00,  2.74s/it]

✅ Generated 442 reports


In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rs

# BLEU
_smooth   = SmoothingFunction().method1
refs_tok  = [[r.split()] for r in all_refs]
hyps_tok  = [h.split()   for h in all_hyps]

bleu2 = corpus_bleu(refs_tok, hyps_tok, weights=(0.5, 0.5, 0, 0),        smoothing_function=_smooth)
bleu3 = corpus_bleu(refs_tok, hyps_tok, weights=(0.33, 0.33, 0.33, 0),   smoothing_function=_smooth)
bleu4 = corpus_bleu(refs_tok, hyps_tok, weights=(0.25, 0.25, 0.25, 0.25),smoothing_function=_smooth)

# ROUGE
scorer = rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
r1, r2, rl = [], [], []
for ref, hyp in zip(all_refs, all_hyps):
    s = scorer.score(ref, hyp)
    r1.append(s["rouge1"].fmeasure)
    r2.append(s["rouge2"].fmeasure)
    rl.append(s["rougeL"].fmeasure)

results = {
    "BLEU-2":   round(bleu2, 4),
    "BLEU-3":   round(bleu3, 4),
    "BLEU-4":   round(bleu4, 4),
    "ROUGE-1":  round(float(np.mean(r1)), 4),
    "ROUGE-2":  round(float(np.mean(r2)), 4),
    "ROUGE-L":  round(float(np.mean(rl)), 4),
}

print("="*40)
print("  RESULTS")
print("="*40)
for k, v in results.items():
    print(f"  {k:<20}: {v:.4f}")
print("="*40)

  RESULTS
  BLEU-2              : 0.0000
  BLEU-3              : 0.0000
  BLEU-4              : 0.0000
  ROUGE-1             : 0.0035
  ROUGE-2             : 0.0000
  ROUGE-L             : 0.0035


In [ ]:
import json
from pathlib import Path
from google.colab import files

all_outputs = {
    "summary_metrics": {
        "num_samples": N,
        **results,
    },
    "predictions": [
        {
            "uid":        all_uids[i],
            "reference":  all_refs[i],
            "prediction": all_hyps[i],
            "rougeL":     round(per_sample_rougeL[i], 4),
        }
        for i in range(N)
    ],
    "checkpoint_info": {
        "epoch":       int(ckpt["epoch"]),
        "best_rougeL": float(ckpt["best_rougeL"]),
    }
}

json_path = Path("/content/METransformer_outputs.json")
with open(json_path, "w") as f:
    json.dump(all_outputs, f, indent=2)

size_mb = json_path.stat().st_size / (1024*1024)
print(f"✅ Saved → {json_path}  ({size_mb:.2f} MB)")

files.download(str(json_path))
print("✅ Download started!")

✅ Saved → /content/METransformer_outputs.json  (0.24 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started!


## Training loop


In [ ]:
best_rouge_l = 0.0
best_ckpt    = CHECKPOINT_DIR / "best_model.pt"
history      = []

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch:02d}/{NUM_EPOCHS}")
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
    val_rl     = evaluate_rougeL(model, val_loader, gpt2_tokenizer)

    history.append({
        "epoch": epoch, "train_loss": round(train_loss, 4), "val_rougeL": round(val_rl, 4)
    })
    print(f"  Loss={train_loss:.4f}  |  Val ROUGE-L={val_rl:.4f}")

    if val_rl > best_rouge_l:
        best_rouge_l = val_rl
        torch.save({
            "epoch": epoch, "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(), "best_rougeL": best_rouge_l,
        }, best_ckpt)
        print(f"  ✅ New best ROUGE-L: {best_rouge_l:.4f}  — saved to Drive")

    time.sleep(0.1)   # flush Drive writes

hist_path = OUTPUT_DIR / "training_history.csv"
pd.DataFrame(history).to_csv(hist_path, index=False)
print(f"\n🎉 Training done. History → {hist_path}")



Epoch 01/10


  Loss=2.1516  |  Val ROUGE-L=0.0007
  ✅ New best ROUGE-L: 0.0007  — saved to Drive

Epoch 02/10


  Loss=0.6988  |  Val ROUGE-L=0.0010
  ✅ New best ROUGE-L: 0.0010  — saved to Drive

Epoch 03/10


  Loss=0.5487  |  Val ROUGE-L=0.0002

Epoch 04/10


  Loss=0.4593  |  Val ROUGE-L=0.0032
  ✅ New best ROUGE-L: 0.0032  — saved to Drive

Epoch 05/10


...

  Loss=0.3267  |  Val ROUGE-L=0.0008

Epoch 07/10


  Loss=0.2797  |  Val ROUGE-L=0.0032

Epoch 08/10


  Loss=0.2445  |  Val ROUGE-L=0.0009

Epoch 09/10


  Loss=0.2230  |  Val ROUGE-L=0.0008

Epoch 10/10


  Loss=0.2149  |  Val ROUGE-L=0.0009

🎉 Training done. History → /content/iu_xray/outputs/training_history.csv


In [ ]:
import json
import torch
import pandas as pd
from pathlib import Path
from google.colab import files

OUTPUT_DIR     = Path("/content/iu_xray/outputs")
CHECKPOINT_DIR = Path("/content/iu_xray/checkpoints")

all_outputs = {}

# Summary metrics
f = OUTPUT_DIR / "summary_metrics.json"
if f.exists():
    with open(f) as fp: all_outputs["summary_metrics"] = json.load(fp)
    print("✅ summary_metrics")

# Lexical metrics
f = OUTPUT_DIR / "lexical_metrics.json"
if f.exists():
    with open(f) as fp: all_outputs["lexical_metrics"] = json.load(fp)
    print("✅ lexical_metrics")

# Clinical metrics
f = OUTPUT_DIR / "clinical_metrics.json"
if f.exists():
    with open(f) as fp: all_outputs["clinical_metrics"] = json.load(fp)
    print("✅ clinical_metrics")

# Calibration metrics
f = OUTPUT_DIR / "calibration_metrics.json"
if f.exists():
    with open(f) as fp: all_outputs["calibration_metrics"] = json.load(fp)
    print("✅ calibration_metrics")

# Training history
f = OUTPUT_DIR / "training_history.csv"
if f.exists():
    all_outputs["training_history"] = pd.read_csv(f).to_dict(orient="records")
    print("✅ training_history")

# Predictions
f = OUTPUT_DIR / "predictions.csv"
if f.exists():
    all_outputs["predictions"] = pd.read_csv(f).to_dict(orient="records")
    print("✅ predictions")

# Label predictions
f = OUTPUT_DIR / "label_predictions.csv"
if f.exists():
    all_outputs["label_predictions"] = pd.read_csv(f).to_dict(orient="records")
    print("✅ label_predictions")

# Checkpoint info
f = CHECKPOINT_DIR / "best_model.pt"
if f.exists():
    ckpt = torch.load(f, map_location="cpu")
    all_outputs["checkpoint_info"] = {
        "epoch":       ckpt["epoch"],
        "best_rougeL": ckpt["best_rougeL"],
        "size_mb":     round(f.stat().st_size / 1e6, 2)
    }
    print("✅ checkpoint_info")

# Save and download
json_path = Path("/content/METransformer_outputs.json")
with open(json_path, "w") as f:
    json.dump(all_outputs, f, indent=2)

size_mb = json_path.stat().st_size / (1024*1024)
print(f"\n✅ Saved → {json_path}  ({size_mb:.2f} MB)")
files.download(str(json_path))
print("✅ Download started!")

✅ training_history
✅ checkpoint_info

✅ Saved → /content/METransformer_outputs.json  (0.00 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started!


## Load best checkpoint & full test evaluation

In [ ]:
print("Loading best checkpoint …")
ckpt = torch.load(best_ckpt, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
print(f"✅ Loaded epoch {ckpt['epoch']}  (val ROUGE-L = {ckpt['best_rougeL']:.4f})")


Loading best checkpoint …
✅ Loaded epoch 4  (val ROUGE-L = 0.0032)


In [ ]:
import json

with open("/content/METransformer_outputs.json") as f:
    data = json.load(f)

print("Keys in your JSON:")
for key, val in data.items():
    if isinstance(val, list):
        print(f"  {key}: {len(val)} records")
    elif isinstance(val, dict):
        print(f"  {key}: {list(val.keys())}")
    else:
        print(f"  {key}: {val}")

Keys in your JSON:
  training_history: 10 records
  checkpoint_info: ['epoch', 'best_rougeL', 'size_mb']


In [ ]:
from google.colab import files

# Download checkpoint to your local machine right now
files.download("/content/iu_xray/checkpoints/best_model.pt")
print("✅ Download started — save it somewhere safe on your computer!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started — save it somewhere safe on your computer!
